# Recherche automatique de sources (Xanto) — exécution sur Colab

Méthode et règles : `PREINSCRIPTION.md` et `PREINSCRIPTION_ETAPE2.md`. Les images (musées) sont sur le Drive privé ; rien n'est publié ici. Environnement : Exécution > Modifier le type d'exécution > GPU T4, puis Exécution > Tout exécuter.

In [ ]:
!nvidia-smi -L || echo 'PAS DE GPU'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
ZIP = '/content/drive/MyDrive/xanto-prive/recherche-sources/xanto-rs.zip'
print(os.path.exists(ZIP))

In [ ]:
!cp "$ZIP" /content/ 2>/dev/null || cp '/content/drive/MyDrive/xanto-prive/recherche-sources/xanto-rs.zip' /content/
!cd /content && unzip -q -o xanto-rs.zip && ls xanto-rs/recherche-sources | head
!rm -rf /content/xanto && git clone -q https://github.com/maribakulj/xanto /content/xanto && cd /content/xanto && git log -1 --format=%h

In [ ]:
# Étape 1 (méthode préinscrite, DINOv2-base)
!cd /content && python xanto/recherche-sources/recherche.py /content/xanto-rs/recherche-sources 2>&1 | grep -v Warning | tail -8

In [ ]:
# Étape 2 (reclassement par détails cohérents)
!cd /content && python xanto/recherche-sources/etape2.py /content/xanto-rs/recherche-sources 2>&1 | grep -v Warning | tail -6

In [ ]:
OUT = '/content/drive/MyDrive/xanto-prive/recherche-sources/resultats'
!mkdir -p "$OUT" && cp /content/xanto-rs/recherche-sources/resultats_*.csv "$OUT"/ && ls "$OUT"

## Exploratoire : DINOv3 (ne compte pas pour la décision)
Nécessite le secret Colab `HF_TOKEN` (icône clé à gauche) d'un compte ayant accepté la licence de `facebook/dinov3-vitb16-pretrain-lvd1689m`.

In [ ]:
from google.colab import userdata
from huggingface_hub import login
login(userdata.get('HF_TOKEN'))
!cd /content && python xanto/recherche-sources/recherche.py /content/xanto-rs/recherche-sources facebook/dinov3-vitb16-pretrain-lvd1689m 2>&1 | grep -v Warning | tail -6
!cp /content/xanto-rs/recherche-sources/resultats_*.csv "$OUT"/